In [ ]:
# ============================================
# Git Cheat Sheet NLP + K-Means Clustering
# ============================================

# Import libraries
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# --------------------------------------------
# Load dataset
# --------------------------------------------

df = pd.read_csv("git-cheat-sheet.csv")

print(df.head())

# --------------------------------------------
# Check missing values
# --------------------------------------------

print(df.isnull().sum())

# Remove rows with missing descriptions
df = df.dropna(subset=["description"])

# --------------------------------------------
# NLP Preprocessing with TF-IDF
# --------------------------------------------

vectorizer = TfidfVectorizer(
    stop_words="english",
    lowercase=True
)

X = vectorizer.fit_transform(df["description"])

print("TF-IDF Shape:", X.shape)

# --------------------------------------------
# K-Means Clustering
# --------------------------------------------

k = 5

kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

df["Cluster"] = kmeans.fit_predict(X)

print(df[["command","Cluster"]].head())

# --------------------------------------------
# Display commands by cluster
# --------------------------------------------

for cluster in sorted(df["Cluster"].unique()):

    print("\n============================")
    print(f"Cluster {cluster}")
    print("============================")

    commands = df[df["Cluster"] == cluster]

    print(commands[["command","description"]])

# --------------------------------------------
# Top keywords in each cluster
# --------------------------------------------

terms = vectorizer.get_feature_names_out()

order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]

print("\nTop Keywords")

for i in range(k):

    print(f"\nCluster {i}")

    keywords = []

    for ind in order_centroids[i, :10]:
        keywords.append(terms[ind])

    print(", ".join(keywords))

# --------------------------------------------
# PCA Visualization
# --------------------------------------------

pca = PCA(n_components=2)

reduced = pca.fit_transform(X.toarray())

plt.figure(figsize=(10,6))

plt.scatter(
    reduced[:,0],
    reduced[:,1],
    c=df["Cluster"],
    cmap="tab10",
    s=80
)

for i, cmd in enumerate(df["command"]):
    plt.text(
        reduced[i,0],
        reduced[i,1],
        cmd,
        fontsize=8
    )

plt.title("Git Commands Clustering")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.grid(True)

plt.show()

# --------------------------------------------
# Cluster Summary
# --------------------------------------------

summary = df.groupby("Cluster").size()

print("\nCluster Sizes")
print(summary)

# --------------------------------------------
# Save clustered dataset
# --------------------------------------------

df.to_csv(
    "git-cheat-sheet-clustered.csv",
    index=False
)

print("\nClustered dataset saved!")